# Faz 3 v3 — HFP Grafting: Qwen2.5-1.5B + O(1) Bellek (Distilasyon)

v2'nin temiz yeniden kurulumu. **Yenilikler (v2 kosusunun dersleri):**
- **Kesinti sigortasi:** Google Drive baglanirsa checkpoint'ler oraya yazilir (oturum dusse de kalir); her 250 adimda kayit.
- **RESUME:** onceki kosudan inen `hfp_graft_stage1_*.pt` / `hfp_graft_stage2_*.pt` dosyasiyla kaldigi yerden devam (Hucre 4).
- **S1 dersi:** ilk kosuda MSE ~1000-1150 adimda 0.07 platosuna oturdu -> `STEPS1=1200` (gerekirse artir).
- GPU assert + NaN korumalari.

**Kosma sirasi:** 1 → 2 → 3 → (checkpoint'in varsa 4) → 5 (S1; resume ettiysen otomatik atlanabilir) → 6 (S2) → 7 (validasyon).

**Sure tahmini (T4):** S1 ~0.83 dk/adim (1200 adim ≈ 16-17 sa; resume ile 0), S2 ilk 50-adim logunda dk/adim gorulur.
**On-kayitli kriterler (SONRAKI_ADIMLAR_PLANI.md K3):** (a) zero-shot PPL<1000, (b) S2 sonrasi PPL ≤ 1.05× orijinal (1.05-1.20 kabul edilebilir ilk gecis), (c) alpha dagilimi kaydi, (d) needle 2K sanity, (e) O(1) state raporu.


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys
import importlib

# [SIGORTA] GPU assert — CPU'da 15 saat bosa kosma dersi
import torch
assert torch.cuda.is_available(), 'GPU SECILI DEGIL! Runtime > Change runtime type > T4 GPU.'
# [SIGORTA] Mimari uyumu: yeni torch derlemeleri P100/Pascal'i (sm_60) desteklemiyor
# ("no kernel image" hatasi). Kaggle'da P100 DEGIL, T4 / T4 x2 secin.
_cap = torch.cuda.get_device_capability(0)
_arch = f'sm_{_cap[0]}{_cap[1]}'
assert _arch in torch.cuda.get_arch_list(), (
    f'GPU mimarisi {_arch} bu torch derlemesinde YOK {torch.cuda.get_arch_list()} '
    f'-> Accelerator olarak T4 secin (P100 desteklenmiyor).')
DEV = 'cuda'

BASE = '/content' if os.path.exists('/content') else ('/kaggle/working' if os.path.exists('/kaggle') else '.')
REPO = os.path.join(BASE, 'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/kayra-hn/HFP.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'transformers>=4.46', 'datasets', 'accelerate'], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)

# [SIGORTA] Checkpoint'ler icin Drive (varsa) — oturum dusse de dosyalar kalir
CKPT_DIR = BASE
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/hfp_graft_ckpt'
    os.makedirs(CKPT_DIR, exist_ok=True)
    print('Checkpoint dizini (Drive):', CKPT_DIR)
except Exception as e:
    print(f'Drive yok/reddedildi ({type(e).__name__}) -> checkpoint {BASE} altina yazilir; ARA ARA INDIR!')

# HF girisi: Qwen2.5-1.5B UNGATED -> token OPSIYONEL. Prompt YOK (arka plan
# kosumlarinda bloklar); varsa Colab/Kaggle Secrets'tan alinir.
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
    print('HF token Colab Secrets\'tan alindi.')
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN').strip()
        print('HF token Kaggle Secrets\'tan alindi.')
    except Exception:
        print('HF token yok — Qwen2.5-1.5B ungated, tokensiz devam.')

print(f'PyTorch {torch.__version__} | {torch.cuda.get_device_name(0)}')

# graft_smoke: torch ortaminda ILK kosum — gecmeden devam etmeyin
env = dict(os.environ, PYTHONPATH=REPO + ':' + os.environ.get('PYTHONPATH', ''))
r = subprocess.run([sys.executable, 'review_scripts/graft_smoke.py'],
                   capture_output=True, text=True, env=env, cwd=REPO)
print(r.stdout[-2500:]); print(r.stderr[-1500:] if r.returncode else '')
assert r.returncode == 0, 'graft_smoke.py BASARISIZ — devam etmeyin'


In [ ]:
# --- 2. MODEL + GRAFT ONCESI REFERANS PPL ---
import math, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

MODEL_ID = 'Qwen/Qwen2.5-1.5B'           # ungated, Apache 2.0

_tk = os.environ.get('HF_TOKEN') or None
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=_tk)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32,
                                             token=_tk).to(DEV)
model.eval()
print(f'{MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e9:.2f}B param')

wiki = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='validation')
val_text = '\n'.join(t for t in wiki['text'] if t.strip())
val_ids = tok(val_text, return_tensors='pt').input_ids[0]
print(f'Val: {len(val_ids):,} token')

@torch.no_grad()
def ppl(model, ids, seq=1024, n_chunks=24):
    model.eval(); losses = []
    for i in range(n_chunks):
        s = i * seq
        if s + seq > len(ids): break
        x = ids[s:s+seq].unsqueeze(0).to(DEV)
        out = model(x, labels=x)   # HF labels'i iceride kaydirir; disaridan kaydirma = cift-kaydirma hatasi
        losses.append(out.loss.item())
    return math.exp(sum(losses)/len(losses))

PPL_BASE = ppl(model, val_ids)
print(f'GRAFT ONCESI referans PPL: {PPL_BASE:.2f}')


In [ ]:
# --- 3. GRAFT + SIFIRINCI-ADIM SANITY ---
from hfp.models.grafting import (GraftConfig, graft_llama, set_graft_mode, set_distill_backward,
                                 distill_loss, trainable_parameters,
                                 enable_streaming, reset_streaming, HFPGraftAttention)

SAVE_KEYS = ('decay','log_eta','conv_q','conv_k','beta_gate','alpha_logit','retrieval_norm','out_gain')
def save_hfp(tag):
    path = f'{CKPT_DIR}/hfp_graft_{tag}.pt'
    torch.save({n: p for n, p in model.state_dict().items()
                if any(k in n for k in SAVE_KEYS)}, path)
    print(f'  checkpoint: {path}', flush=True)

gcfg = GraftConfig(decay_mode='cubic_flux_chunked', write_rule='hybrid',
                   key_feature_map='dpfp', rec_block=16)   # [VRAM] T4: kucuk blok sart
GRAFT_LAYERS = graft_llama(model, gcfg)      # default: tek-indeksli katmanlar (yari-yariya hibrit)

# teacher modu == orijinal model (dogrulama) — AYNI chunk sayisiyla kiyasla!
set_graft_mode(model, 'teacher')
ppl_teacher = ppl(model, val_ids)            # PPL_BASE ile ayni 24 chunk
print(f'teacher {ppl_teacher:.2f} vs base {PPL_BASE:.2f}')
assert abs(ppl_teacher - PPL_BASE) < PPL_BASE*0.01, 'teacher yolu bozuk!'

# student modu, egitimsiz: PPL kotu olacak ama COP OLMAMALI (kriter: <1000)
set_graft_mode(model, 'student')
PPL_ZERO = ppl(model, val_ids, n_chunks=8)
print(f'Zero-shot (egitimsiz HFP) PPL: {PPL_ZERO:.1f}  (kriter <1000: {"GECTI" if PPL_ZERO < 1000 else "KALDI — agirlik transferi/olcek kontrol"})')


In [ ]:
# --- 4. RESUME (OPSIYONEL): onceki kosunun checkpoint'inden devam ---
# Kullanim: .pt dosyasini sol Files paneline (veya Drive'daki hfp_graft_ckpt/)
# surukle; RESUME_PATH bos birakilirsa en yenisi otomatik bulunur.
import glob, torch

RESUME_PATH = ''          # orn: '/content/hfp_graft_stage1_kesildi_1150.pt' — bos = otomatik ara
SKIP_S1 = False           # resume stage1>=1000 ise asagida otomatik True yapilir

if not RESUME_PATH:
    cands = sorted(glob.glob(f'{CKPT_DIR}/hfp_graft_*.pt') + glob.glob(f'{BASE}/hfp_graft_*.pt')
                   + glob.glob('/kaggle/input/*/hfp_graft_*.pt'),      # Kaggle: onceki versiyon ciktisi
                   key=os.path.getmtime)
    RESUME_PATH = cands[-1] if cands else ''

if RESUME_PATH:
    sd = torch.load(RESUME_PATH, map_location=DEV)
    res = model.load_state_dict(sd, strict=False)
    n_loaded = len(sd) - len(res.unexpected_keys)
    assert n_loaded > 0 and not res.unexpected_keys, f'checkpoint uyumsuz: {res.unexpected_keys[:5]}'
    alphas = [torch.sigmoid(m.alpha_logit).mean().item()
              for m in model.modules() if isinstance(m, HFPGraftAttention)]
    print(f'RESUME: {RESUME_PATH}  ({n_loaded} tensor yuklendi)  alpha_ort {sum(alphas)/len(alphas):.3f}')
    if 'stage1' in RESUME_PATH:
        import re
        m = re.search(r'(\d+)', RESUME_PATH.split('stage1')[-1])
        if m and int(m.group(1)) >= 700:   # v2+v3: MSE ~750-800'de plato (0.087-0.088)
            SKIP_S1 = True
            print('-> S1 yeterince kosulmus: SKIP_S1=True (Hucre 5 atlanacak, dogrudan Hucre 6/S2).')
    if 'stage2' in RESUME_PATH or 'final' in RESUME_PATH:
        SKIP_S1 = True
        print('-> S2 checkpoint: Hucre 5 atlanir; S2 kalan adimlar icin STEPS2 dusurulebilir.')
else:
    print('Checkpoint bulunamadi — sifirdan S1 (Hucre 5) kosulacak.')


In [ ]:
# --- 5. STAGE 1: TEACHER-FORCING DISTILASYON (katman-ici MSE) ---
# Ileriye ogretmen ciktisi gider -> katmanlar bagimsiz, akis on-distribution.
# [v3] Ilk kosu dersi: MSE ~1000-1150 adimda 0.07 platosuna oturdu -> STEPS1=1200.
import time
from datasets import load_dataset
from transformers import get_cosine_schedule_with_warmup

SEQ, BATCH, STEPS1, LR1 = 1024, 1, 1200, 1e-3
ACCUM = 4

train_stream = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1',
                            split='train', streaming=True)

def batch_iter(stream, seq, batch):
    buf = []
    for ex in stream:
        if not ex['text'].strip(): continue
        buf.extend(tok(ex['text']).input_ids)
        while len(buf) >= batch * (seq + 1):
            chunk = buf[:batch*(seq+1)]; buf = buf[batch*(seq+1):]
            t = torch.tensor(chunk).view(batch, seq+1)
            yield t[:, :-1], t[:, 1:]

it = batch_iter(train_stream, SEQ, BATCH)

if globals().get('SKIP_S1', False):
    print('SKIP_S1=True -> Stage 1 atlandi (resume edilen agirliklarla S2ye gecin).')
else:
    model.config.use_cache = False   # egitimde cache kapali
    opt = torch.optim.AdamW(trainable_parameters(model), lr=LR1, weight_decay=0.01)
    sch = get_cosine_schedule_with_warmup(opt, 100, STEPS1)
    set_graft_mode(model, 'teacher_forcing')
    set_distill_backward(model, 1.0 / ACCUM)   # [VRAM] katman-aninda backward
    model.train()

    t0 = time.time(); log = []
    for step in range(1, STEPS1 + 1):
        loss_acc = 0.0
        for _ in range(ACCUM):
            x, _ = next(it)
            model(x.to(DEV))                    # backward katman icinde otomatik calisir
            aux = distill_loss(model)           # detached — yalnizca loglama
            loss_acc += aux.item() / ACCUM
        assert loss_acc == loss_acc, f'NaN @ S1 step {step}'   # NaN korumasi
        torch.nn.utils.clip_grad_norm_(trainable_parameters(model), 1.0)
        opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        if step % 50 == 0 or step <= 3:   # ilk adimlar hemen loglanir ("takildi mi?" suphesine karsi)
            alphas = [torch.sigmoid(m.alpha_logit).mean().item()
                      for m in model.modules() if isinstance(m, HFPGraftAttention)]
            print(f'S1 {step}/{STEPS1}  mse {loss_acc:.5f}  alpha_ort {sum(alphas)/len(alphas):.3f}  {(time.time()-t0)/60:.0f}dk', flush=True)
            log.append((step, loss_acc))
        if step % 250 == 0:
            save_hfp(f'stage1_{step}')          # [SIGORTA] Drive'a / BASE'e
    save_hfp('stage1_son')
    print('Stage 1 tamam.')


In [ ]:
# --- 6. STAGE 2: LOGIT-KL + LM LOSS (uctan uca) ---
# Ogretmen logitleri ayni modelin "teacher" modundan (ikinci kopya yok).
import time
import torch.nn.functional as F

STEPS2, LR2, KL_W, TEMP = 1000, 3e-4, 0.5, 2.0
# [VRAM] Qwen vocab 151k -> logit tensorleri buyuk; checkpointing T4 icin sart
set_distill_backward(model, None)   # Stage 2'de kapali (uctan uca graf)
model.gradient_checkpointing_enable()
model.config.use_cache = False
opt = torch.optim.AdamW(trainable_parameters(model), lr=LR2, weight_decay=0.01)
sch = get_cosine_schedule_with_warmup(opt, 50, STEPS2)

t0 = time.time()
for step in range(1, STEPS2 + 1):
    loss_acc = 0.0
    for _ in range(ACCUM):
        x, y = next(it)
        x = x.to(DEV)
        with torch.no_grad():
            set_graft_mode(model, 'teacher')
            t_logits = model(x).logits
        set_graft_mode(model, 'student')
        out = model(x, labels=x)   # HF ic kaydirma (dogru next-token)
        kl = F.kl_div(F.log_softmax(out.logits/TEMP, -1),
                      F.log_softmax(t_logits/TEMP, -1),
                      log_target=True, reduction='batchmean') * TEMP * TEMP
        loss = out.loss + KL_W * kl
        assert torch.isfinite(loss), f'NaN/Inf @ S2 step {step}'
        (loss / ACCUM).backward()
        loss_acc += loss.item() / ACCUM
    torch.nn.utils.clip_grad_norm_(trainable_parameters(model), 1.0)
    opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
    if step % 50 == 0 or step <= 3:
        print(f'S2 {step}/{STEPS2}  loss {loss_acc:.4f}  lm {out.loss.item():.4f}  {(time.time()-t0)/60:.0f}dk', flush=True)
    if step % 250 == 0:
        save_hfp(f'stage2_{step}')              # [SIGORTA]
save_hfp('final')
print('Stage 2 tamam; HFP parametreleri kaydedildi.')


In [ ]:
# --- 7. VALIDASYON: PPL + NEEDLE + VRAM ---
import math, random, torch

model.gradient_checkpointing_disable()
model.config.use_cache = True
# 6a. Kisa-baglam PPL kaybi (kriter: <= 1.05x orijinal)
set_graft_mode(model, 'student')
PPL_FINAL = ppl(model, val_ids)
print(f'PPL: orijinal {PPL_BASE:.2f} -> HFP-hibrit {PPL_FINAL:.2f} '
      f'({PPL_FINAL/PPL_BASE:.3f}x; kriter <=1.05x: {"GECTI" if PPL_FINAL <= 1.05*PPL_BASE else "KALDI"})')

# 6b. Igne-deligi (needle) — O(1) streaming: grafted state tasinir,
# full-attn katmanlar icin HF cache kullanilir.
from transformers import DynamicCache

def needle_test(model, tok, total_tokens=8192, chunk=512, seed=0):
    random.seed(seed)
    secret = random.choice(['blue falcon', 'copper mountain', 'silent river'])
    needle = f' The secret passphrase is {secret}. '
    filler = ' The weather was ordinary and nothing of note happened that day.'
    fill_ids = tok(filler, add_special_tokens=False).input_ids
    text_ids = []
    while len(text_ids) < total_tokens: text_ids.extend(fill_ids)
    ins = len(text_ids) // 8                      # igne basa yakin -> uzun gap
    needle_ids = tok(needle, add_special_tokens=False).input_ids
    text_ids = text_ids[:ins] + needle_ids + text_ids[ins:total_tokens - len(needle_ids)]
    query_ids = tok(' The secret passphrase is', add_special_tokens=False).input_ids
    ids = torch.tensor(text_ids + query_ids).unsqueeze(0).to(DEV)

    set_graft_mode(model, 'student'); model.eval()
    enable_streaming(model, True); reset_streaming(model)
    cache = DynamicCache()
    with torch.no_grad():
        for s in range(0, ids.size(1), chunk):
            out = model(ids[:, s:s+chunk], past_key_values=cache, use_cache=True)
            cache = out.past_key_values
        gen = []
        last = out.logits[:, -1:].argmax(-1)
        for _ in range(6):
            gen.append(last.item())
            out = model(last, past_key_values=cache, use_cache=True)
            cache = out.past_key_values
            last = out.logits[:, -1:].argmax(-1)
    enable_streaming(model, False)
    answer = tok.decode(gen)
    hit = secret.split()[0] in answer
    print(f'  L={total_tokens}: gizli="{secret}" cevap="{answer.strip()}" -> {"BULDU" if hit else "BULAMADI"}')
    return hit

print('Needle (igne) testi:')
for L in [2048, 8192, 16384]:
    needle_test(model, tok, total_tokens=L)

# 6c. VRAM / state sabitligi: grafted katman state boyutu tokenle degismemeli
attn = next(m for m in model.modules() if isinstance(m, HFPGraftAttention))
if attn._stream_state is not None:
    M, z = attn._stream_state[0], attn._stream_state[1]
    print(f'Grafted state: M {tuple(M.shape)} + z {tuple(z.shape)} '
          f'= {(M.numel()+z.numel())*4/1e6:.1f} MB — baglam uzunlugundan BAGIMSIZ (O(1))')
if DEV == 'cuda':
    print(f'Tepe VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
print('\nBitince: GPU_ROADMAP.md §10 checklist + RESULTS.md guncellenecek.')